## 1. Configuration et Imports

Import des bibliotheques necessaires et configuration des chemins de travail.

In [1]:
import pandas as pd
import numpy as np
import os
import re
from datetime import datetime
from typing import Dict, List, Tuple, Optional
import warnings
warnings.filterwarnings('ignore')

# Tentative d'import de pdfplumber pour l'extraction PDF
try:
    import pdfplumber
    PDF_SUPPORT = True
    print("pdfplumber disponible - Extraction PDF activee")
except ImportError:
    PDF_SUPPORT = False
    print("pdfplumber non disponible - Installation requise: pip install pdfplumber")

# Configuration des chemins
BASE_PATH = '/home/henintsoa/CFIM'
METEO_PATH = os.path.join(BASE_PATH, 'meteo')
CSV_PATH = os.path.join(BASE_PATH, 'csv')
OUTPUT_PATH = os.path.join(BASE_PATH, 'final/data')

print(f"\nChemins configures:")
print(f"   Base: {BASE_PATH}")
print(f"   Meteo: {METEO_PATH}")
print(f"   CSV: {CSV_PATH}")
print(f"   Output: {OUTPUT_PATH}")

pdfplumber disponible - Extraction PDF activee

Chemins configures:
   Base: /home/henintsoa/CFIM
   Meteo: /home/henintsoa/CFIM/meteo
   CSV: /home/henintsoa/CFIM/csv
   Output: /home/henintsoa/CFIM/final/data


## 2. Inventaire des Bulletins Disponibles

Comptage des bulletins meteo disponibles par annee avant l'extraction.

In [2]:
def inventorier_bulletins_meteo(meteo_path: str) -> pd.DataFrame:
    """
    Parcourt tous les dossiers meteo et compte les bulletins disponibles.
    
    Args:
        meteo_path: Chemin vers le repertoire des donnees meteo
    
    Returns:
        DataFrame avec le nombre de bulletins par annee et type
    """
    inventaire = []
    
    # Parcourir les annees
    for year_folder in sorted(os.listdir(meteo_path)):
        year_path = os.path.join(meteo_path, year_folder)
        
        # Filtrer uniquement les dossiers METEO
        if not os.path.isdir(year_path) or not year_folder.startswith('METEO'):
            continue
        
        # Extraire l'annee du nom du dossier
        year_match = re.search(r'(\d{4})', year_folder)
        if not year_match:
            continue
        year = year_match.group(1)
        
        # Compteurs pour cette annee
        nb_jours = 0
        nb_pdf_marine_cotiere = 0
        nb_pdf_marine_hm = 0
        nb_pdf_total = 0
        
        # Parcourir les sous-dossiers (jours)
        for day_folder in os.listdir(year_path):
            day_path = os.path.join(year_path, day_folder)
            
            if not os.path.isdir(day_path):
                continue
            
            nb_jours += 1
            
            # Compter les PDF par type
            for file in os.listdir(day_path):
                if file.endswith('.pdf'):
                    nb_pdf_total += 1
                    # Identifier le type de bulletin
                    if 'cotiere' in file.lower():
                        nb_pdf_marine_cotiere += 1
                    elif 'marine' in file.lower() and 'hm' in file.lower():
                        nb_pdf_marine_hm += 1
        
        # Ajouter a l'inventaire
        inventaire.append({
            'Annee': year,
            'Jours disponibles': nb_jours,
            'PDF Marine Cotiere': nb_pdf_marine_cotiere,
            'PDF Marine HM': nb_pdf_marine_hm,
            'Total PDF': nb_pdf_total
        })
    
    return pd.DataFrame(inventaire)

# Executer l'inventaire
df_inventaire = inventorier_bulletins_meteo(METEO_PATH)

print("INVENTAIRE DES BULLETINS METEO DISPONIBLES")
print("=" * 70)
print(df_inventaire.to_string())

# Statistiques globales
total_jours = df_inventaire['Jours disponibles'].sum()
total_pdf_cotiere = df_inventaire['PDF Marine Cotiere'].sum()
print(f"\nTOTAL:")
print(f"   {total_jours} jours de donnees")
print(f"   {total_pdf_cotiere} bulletins Marine Cotiere a extraire")

INVENTAIRE DES BULLETINS METEO DISPONIBLES
  Annee  Jours disponibles  PDF Marine Cotiere  PDF Marine HM  Total PDF
0  2019                103                 177             60        334
1  2020                 88                 131             19        295
2  2021                161                 229            147        518
3  2022                181                 253            184        801
4  2023                133                 183             85        369
5  2024                  0                   0              0          0
6  2025                 23                  24             27         83

TOTAL:
   689 jours de donnees
   997 bulletins Marine Cotiere a extraire


**Resultat attendu:** Tableau listant le nombre de bulletins disponibles par annee (2019-2022)

## 3. Classe d'Extraction des Bulletins Meteo

Cette classe contient toute la logique pour:
1. Lire les fichiers PDF des bulletins marins
2. Parser le texte pour extraire les informations structurees
3. Identifier les zones cotieres et leurs conditions meteo

In [3]:
class BulletinMeteoExtractor:
    """
    Extracteur de donnees meteorologiques marines depuis les bulletins PDF.
    
    Cette classe parse les bulletins Marine Cotiere de Madagascar et extrait:
    - Les zones cotieres concernees
    - Les conditions de vent (direction, vitesse)
    - L'etat de la mer
    - Les conditions meteo generales
    """
    
    def __init__(self):
        # Ces zones sont utilisees dans les bulletins marins officiels
        self.zones_cotieres = [
            "CAP D'AMBRE A TOAMASINA",
            "CAP D'AMBRE A MAHANORO",
            "CAP D'AMBRE A ANTALAHA",
            "TOAMASINA AU CAP SAINTE MARIE",
            "TOAMASINA A TAOLAGNARO",
            "MAHANORO AU CAP SAINTE MARIE",
            "CAP D'AMBRE A BESALAMPY",
            "BESALAMPY A MOROMBE",
            "MOROMBE AU CAP SAINTE MARIE",
            "MOROMBE A TAOLAGNARO",
            "CAP D'AMBRE A CAP EST",
            "CAP EST A TAOLAGNARO"
        ]
        
        # Ces patterns permettent d'extraire les differents champs des bulletins
        
        self.pattern_section = r"([A-Z\s'ÀÉÈÊ\-]+(?:A|AU)\s+[A-Z\s'ÀÉÈÊ\-]+)\s*\n"
        self.pattern_vent = r"VENT\s*[:\s]*(.+?)(?=ETAT|MER|TEMPS|HOULE|$)"
        self.pattern_mer = r"(?:ETAT DE LA MER|MER)\s*[:\s]*(.+?)(?=TEMPS|HOULE|VENT|$)"
        self.pattern_temps = r"TEMPS\s*[:\s]*(.+?)(?=VENT|ETAT|MER|HOULE|$)"
        self.pattern_houle = r"HOULE\s*[:\s]*(.+?)(?=VENT|ETAT|MER|TEMPS|$)"
    
    def extraire_date_depuis_nom(self, nom_fichier: str) -> Optional[str]:
        """
        Extrait la date depuis le nom du fichier ou du dossier.
        
        Formats supportes:
        - DDMMYYYY (ex: 01032022)
        - YYYY-MM-DD (ex: 2019-09-03)
        
        Args:
            nom_fichier: Nom du fichier ou dossier contenant la date
        
        Returns:
            Date au format DD/MM/YYYY ou None si non trouve
        """
        # Format DDMMYYYY (ex: 01032022)
        match = re.search(r'(\d{2})(\d{2})(\d{4})', nom_fichier)
        if match:
            jour, mois, annee = match.groups()
            return f"{jour}/{mois}/{annee}"
        
        # Format YYYY-MM-DD (ex: 2019-09-03)
        match = re.search(r'(\d{4})-(\d{2})-(\d{2})', nom_fichier)
        if match:
            annee, mois, jour = match.groups()
            return f"{jour}/{mois}/{annee}"
        
        return None
    
    def lire_pdf(self, pdf_path: str) -> Optional[str]:
        """
        Lit le contenu textuel d'un fichier PDF en utilisant pdfplumber.
        
        Args:
            pdf_path: Chemin complet vers le fichier PDF
            
        Returns:
            Texte extrait du PDF ou None en cas d'erreur
        """
        if not PDF_SUPPORT:
            return None
        
        try:
            with pdfplumber.open(pdf_path) as pdf:
                texte_complet = ""
                # Extraire le texte de chaque page
                for page in pdf.pages:
                    texte = page.extract_text()
                    if texte:
                        texte_complet += texte + "\n"
                return texte_complet
        except Exception as e:
            print(f"   Erreur lecture PDF {pdf_path}: {e}")
            return None
    
    def lire_texte_extrait(self, folder_path: str) -> Optional[str]:
        """
        Lit le texte depuis le dossier extracted_text/ si disponible.
        Cette methode est utilisee quand le texte a deja ete extrait des PDF.
        
        Args:
            folder_path: Chemin vers le dossier du jour
            
        Returns:
            Texte extrait ou None si non disponible
        """
        extracted_path = os.path.join(folder_path, 'extracted_text')
        
        if not os.path.exists(extracted_path):
            return None
        
        texte_complet = ""
        # Concatener tous les fichiers .txt du dossier
        for file in os.listdir(extracted_path):
            if file.endswith('.txt'):
                with open(os.path.join(extracted_path, file), 'r', encoding='utf-8', errors='ignore') as f:
                    texte_complet += f.read() + "\n"
        
        return texte_complet if texte_complet else None
    
    def parser_bulletin(self, texte: str, date_str: str) -> List[Dict]:
        """
        Parse le texte d'un bulletin meteo et extrait les donnees par zone.
        
        Args:
            texte: Texte complet du bulletin
            date_str: Date au format DD/MM/YYYY
            
        Returns:
            Liste de dictionnaires avec les donnees par zone cotiere
        """
        resultats = []
        
        # Normaliser le texte (majuscules, espaces)
        texte = texte.upper()
        texte = re.sub(r'\s+', ' ', texte)
        
        # Extraire les donnees pour chaque zone cotiere
        for zone in self.zones_cotieres:
            zone_upper = zone.upper()
            
            # Verifier si la zone est mentionnee dans le bulletin
            if zone_upper in texte:
                pos = texte.find(zone_upper)
                
                end_pos = len(texte)
                for autre_zone in self.zones_cotieres:
                    if autre_zone.upper() != zone_upper:
                        autre_pos = texte.find(autre_zone.upper(), pos + len(zone_upper))
                        if autre_pos > pos and autre_pos < end_pos:
                            end_pos = autre_pos
                
                bloc = texte[pos:end_pos]
                
                # Extraire les champs meteo du bloc
                vent = self._extraire_champ(bloc, r"VENT\s*[:\s]*(.+?)(?=ETAT|MER|TEMPS|HOULE|[A-Z']+\s+A\s|$)")
                etat_mer = self._extraire_champ(bloc, r"(?:ETAT DE LA MER|MER)\s*[:\s]*(.+?)(?=TEMPS|HOULE|VENT|[A-Z']+\s+A\s|$)")
                temps = self._extraire_champ(bloc, r"TEMPS\s*[:\s]*(.+?)(?=VENT|ETAT|MER|HOULE|[A-Z']+\s+A\s|$)")
                houle = self._extraire_champ(bloc, r"HOULE\s*[:\s]*(.+?)(?=VENT|ETAT|MER|TEMPS|[A-Z']+\s+A\s|$)")
                
                # Ajouter aux resultats si au moins un champ est trouve
                if vent or etat_mer or temps:
                    resultats.append({
                        'date': date_str,
                        'zone': zone,
                        'vent': vent if vent else 'Non specifie',
                        'etat_mer': etat_mer if etat_mer else 'Non specifie',
                        'temps': temps if temps else 'Non specifie',
                        'houle': houle if houle else 'Non specifie'
                    })
        
        return resultats
    
    def _extraire_champ(self, texte: str, pattern: str) -> Optional[str]:
        """
        Extrait un champ specifique du texte avec un pattern regex.
        
        Args:
            texte: Texte a analyser
            pattern: Expression reguliere pour extraire le champ
        
        Returns:
            Valeur extraite ou None
        """
        match = re.search(pattern, texte, re.IGNORECASE | re.DOTALL)
        if match:
            valeur = match.group(1).strip()
            # Nettoyer la valeur (espaces multiples, ponctuation en fin)
            valeur = re.sub(r'\s+', ' ', valeur)
            valeur = valeur.strip('., ')
            return valeur[:200]  # Limiter la longueur pour eviter les erreurs
        return None

print("Classe BulletinMeteoExtractor chargee")

Classe BulletinMeteoExtractor chargee


## 4. Extraction Massive des Bulletins

Parcours de tous les dossiers meteo et extraction des donnees de chaque bulletin.

In [4]:
def extraire_tous_bulletins(meteo_path: str, annees: List[str] = None) -> pd.DataFrame:
    """
    Extrait les donnees de tous les bulletins meteo disponibles.
    
    Args:
        meteo_path: Chemin vers le dossier meteo/
        annees: Liste des annees a traiter (None = toutes les annees)
        
    Returns:
        DataFrame avec toutes les donnees extraites
    """
    extracteur = BulletinMeteoExtractor()
    tous_resultats = []
    
    # Statistiques d'extraction
    stats = {
        'dossiers_traites': 0,
        'pdf_lus': 0,
        'textes_extraits': 0,
        'zones_extraites': 0,
        'erreurs': 0
    }
    
    # Parcourir les annees
    for year_folder in sorted(os.listdir(meteo_path)):
        year_path = os.path.join(meteo_path, year_folder)
        
        # Filtrer uniquement les dossiers METEO
        if not os.path.isdir(year_path) or not year_folder.startswith('METEO'):
            continue
        
        # Filtrer par annee si specifie
        year_match = re.search(r'(\d{4})', year_folder)
        if year_match:
            year = year_match.group(1)
            if annees and year not in annees:
                continue
        
        print(f"\nTraitement de {year_folder}...")
        
        # Parcourir les jours
        for day_folder in sorted(os.listdir(year_path)):
            day_path = os.path.join(year_path, day_folder)
            
            if not os.path.isdir(day_path):
                continue
            
            stats['dossiers_traites'] += 1
            
            # Extraire la date du nom du dossier
            date_str = extracteur.extraire_date_depuis_nom(day_folder)
            if not date_str:
                continue
            
            texte = None
            
            # Methode 1: Lire le texte deja extrait (plus rapide)
            texte = extracteur.lire_texte_extrait(day_path)
            if texte:
                stats['textes_extraits'] += 1
            
            # Methode 2: Lire directement le PDF si texte non disponible
            if not texte and PDF_SUPPORT:
                for file in os.listdir(day_path):
                    if 'cotiere' in file.lower() and file.endswith('.pdf'):
                        pdf_path = os.path.join(day_path, file)
                        texte = extracteur.lire_pdf(pdf_path)
                        if texte:
                            stats['pdf_lus'] += 1
                            break
            
            # Parser le bulletin et extraire les donnees
            if texte:
                try:
                    resultats = extracteur.parser_bulletin(texte, date_str)
                    tous_resultats.extend(resultats)
                    stats['zones_extraites'] += len(resultats)
                except Exception as e:
                    stats['erreurs'] += 1
        
        print(f"   {stats['dossiers_traites']} dossiers traites jusqu'ici")
    
    # Afficher les statistiques finales
    print("\n" + "=" * 70)
    print("STATISTIQUES D'EXTRACTION")
    print("=" * 70)
    for key, value in stats.items():
        print(f"   {key.replace('_', ' ').title()}: {value}")
    
    return pd.DataFrame(tous_resultats)

print("Fonction d'extraction globale prete")

Fonction d'extraction globale prete


In [5]:
print("DEMARRAGE DE L'EXTRACTION DES BULLETINS METEO")
print("=" * 70)
print("Cette operation peut prendre plusieurs minutes...\n")

# Extraire les donnees de 2019 a 2022
# (2017-2018 ne semblent pas avoir de donnees structurees)
df_meteo_extrait = extraire_tous_bulletins(METEO_PATH, annees=['2019', '2020', '2021', '2022'])

print(f"\nExtraction terminee!")
print(f"{len(df_meteo_extrait)} enregistrements extraits")

DEMARRAGE DE L'EXTRACTION DES BULLETINS METEO
Cette operation peut prendre plusieurs minutes...


Traitement de METEO 2019...
   103 dossiers traites jusqu'ici

Traitement de METEO 2020...
   103 dossiers traites jusqu'ici

Traitement de METEO 2020...
   191 dossiers traites jusqu'ici

Traitement de METEO 2021...
   191 dossiers traites jusqu'ici

Traitement de METEO 2021...
   352 dossiers traites jusqu'ici

Traitement de METEO 2022...
   352 dossiers traites jusqu'ici

Traitement de METEO 2022...
   533 dossiers traites jusqu'ici

STATISTIQUES D'EXTRACTION
   Dossiers Traites: 533
   Pdf Lus: 495
   Textes Extraits: 1
   Zones Extraites: 886
   Erreurs: 0

Extraction terminee!
886 enregistrements extraits
   533 dossiers traites jusqu'ici

STATISTIQUES D'EXTRACTION
   Dossiers Traites: 533
   Pdf Lus: 495
   Textes Extraits: 1
   Zones Extraites: 886
   Erreurs: 0

Extraction terminee!
886 enregistrements extraits


**Resultat attendu:** DataFrame contenant plusieurs milliers d'enregistrements (date, zone, vent, etat_mer, temps)

## 5. Apercu des Donnees Extraites

Verification de la qualite et de la repartition des donnees extraites.

In [6]:
if len(df_meteo_extrait) > 0:
    print("APERCU DES DONNEES EXTRAITES")
    print("=" * 70)
    
    print(f"\nDimensions: {df_meteo_extrait.shape[0]} lignes × {df_meteo_extrait.shape[1]} colonnes")
    print(f"\nColonnes: {list(df_meteo_extrait.columns)}")
    
    print("\nEchantillon des donnees:")
    print(df_meteo_extrait.head(15).to_string())
    
    # Statistiques par zone
    print("\nREPARTITION PAR ZONE COTIERE")
    print("-" * 50)
    print(df_meteo_extrait['zone'].value_counts().to_string())
    
    # Statistiques temporelles
    df_meteo_extrait['date_parsed'] = pd.to_datetime(df_meteo_extrait['date'], format='%d/%m/%Y', errors='coerce')
    df_meteo_extrait['annee'] = df_meteo_extrait['date_parsed'].dt.year
    
    print("\nREPARTITION PAR ANNEE")
    print("-" * 50)
    print(df_meteo_extrait['annee'].value_counts().sort_index().to_string())
else:
    print("Aucune donnee extraite.")
    print("Verifiez que pdfplumber est installe: pip install pdfplumber")

APERCU DES DONNEES EXTRAITES

Dimensions: 886 lignes × 6 colonnes

Colonnes: ['date', 'zone', 'vent', 'etat_mer', 'temps', 'houle']

Echantillon des donnees:
          date                          zone                                                                                                                                    vent                                                                                          etat_mer                  temps                            houle
0   07/09/2019        CAP D'AMBRE A MAHANORO                                                                              DE SUD-EST 15/20 KT ATTEIGNANT 25/30 KT AU NORD D'ANTALAHA                                                         AGITÉE À FORTE. HAUTEUR DE VAGUE 2.8/3.2M           Non specifie                       DE SUD-EST
1   07/09/2019  MAHANORO AU CAP SAINTE MARIE                                                                                          DE SUD-EST 20/25 KT ATTEIGNANT LOCALEM

**Resultat attendu:** Affichage de la repartition par zone et par annee, verification de la coherence

## 6. Consolidation avec les Donnees Existantes

Fusion des nouvelles donnees extraites avec le fichier CSV existant `marine_cotiere_2019_2020.csv`.

In [7]:
# Charger le fichier CSV existant (donnees de 2019-2020)
csv_existant = os.path.join(CSV_PATH, 'marine_cotiere_2019_2020.csv')
df_existant = pd.read_csv(csv_existant)

print("DONNEES EXISTANTES")
print("=" * 70)
print(f"Fichier: {csv_existant}")
print(f"Nombre d'enregistrements: {len(df_existant)}")
print(f"Colonnes: {list(df_existant.columns)}")

print("\nApercu:")
print(df_existant.head().to_string())

DONNEES EXISTANTES
Fichier: /home/henintsoa/CFIM/csv/marine_cotiere_2019_2020.csv
Nombre d'enregistrements: 1256
Colonnes: ['date', 'zone', 'vent', 'etat_mer', 'temps']

Apercu:
         date                          zone                                                                             vent                                                                    etat_mer                        temps
0  06/09/2019        CAP D'AMBRE A MAHANORO                                  10/15 kt atteignant 20/25 kt au nord d'Antalaha                                                              agitée à forte      Pluies faible a modérée
1  06/09/2019  MAHANORO AU CAP SAINTE MARIE  05/10 kt devenant progressivement secteur sud 20/25kt localement 30kt vers midi                                                              agitée à forte                       pluies
2  06/09/2019       CAP D'AMBRE A BESALAMPY           15/20 kt localement 20 kt au sud de Majunga, variable 05/10kt ailleurs        

In [8]:
if len(df_meteo_extrait) > 0:
    # Harmoniser les colonnes (garder uniquement les colonnes communes)
    df_nouveau = df_meteo_extrait[['date', 'zone', 'vent', 'etat_mer', 'temps']].copy()
    
    # Concatener les deux DataFrames
    df_complet = pd.concat([df_existant, df_nouveau], ignore_index=True)
    
    # Supprimer les doublons (meme date et zone)
    # On garde la premiere occurrence (donnees existantes prioritaires)
    df_complet = df_complet.drop_duplicates(subset=['date', 'zone'], keep='first')
    
    # Trier par date
    df_complet['date_parsed'] = pd.to_datetime(df_complet['date'], format='%d/%m/%Y', errors='coerce')
    df_complet = df_complet.sort_values('date_parsed').reset_index(drop=True)
    
    print("FUSION DES DONNEES")
    print("=" * 70)
    print(f"Donnees existantes: {len(df_existant)} enregistrements")
    print(f"Nouvelles donnees: {len(df_nouveau)} enregistrements")
    print(f"Apres fusion (sans doublons): {len(df_complet)} enregistrements")
    
    # Statistiques temporelles apres fusion
    df_complet['annee'] = df_complet['date_parsed'].dt.year
    print("\nREPARTITION PAR ANNEE (apres fusion):")
    print(df_complet['annee'].value_counts().sort_index().to_string())
else:
    # Si pas de nouvelles donnees, utiliser uniquement les existantes
    df_complet = df_existant.copy()
    print("Utilisation des donnees existantes uniquement")

FUSION DES DONNEES
Donnees existantes: 1256 enregistrements
Nouvelles donnees: 886 enregistrements
Apres fusion (sans doublons): 1346 enregistrements

REPARTITION PAR ANNEE (apres fusion):
annee
2019    498
2020    535
2021    142
2022    171


**Resultat attendu:** Dataset consolide avec environ 1000-2000 enregistrements couvrant 2019-2022

## 7. Sauvegarde des Donnees Consolidees

Export du fichier final `meteo_marine_cotiere_complet.csv`

In [9]:
# Colonnes finales a sauvegarder
colonnes_finales = ['date', 'zone', 'vent', 'etat_mer', 'temps']

# Creer une version propre pour la sauvegarde (sans colonnes temporaires)
df_final = df_complet[colonnes_finales].copy()

# Chemin de sortie
output_file = os.path.join(OUTPUT_PATH, 'meteo_marine_cotiere_complet.csv')

# Sauvegarder en CSV
df_final.to_csv(output_file, index=False, encoding='utf-8')

print("SAUVEGARDE EFFECTUEE")
print("=" * 70)
print(f"Fichier: {output_file}")
print(f"Nombre d'enregistrements: {len(df_final)}")
print(f"Taille: {os.path.getsize(output_file) / 1024:.1f} KB")

print("\nPhase 1.1 terminee avec succes!")

SAUVEGARDE EFFECTUEE
Fichier: /home/henintsoa/CFIM/final/data/meteo_marine_cotiere_complet.csv
Nombre d'enregistrements: 1346
Taille: 182.1 KB

Phase 1.1 terminee avec succes!


**Fichier produit:** `/home/henintsoa/CFIM/final/data/meteo_marine_cotiere_complet.csv`

**Structure:**
- `date`: Date du bulletin (DD/MM/YYYY)
- `zone`: Zone cotiere concernee
- `vent`: Conditions de vent (texte)
- `etat_mer`: Etat de la mer (texte)
- `temps`: Conditions meteorologiques (texte)